In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.layers import Dense, Embedding, Concatenate, Flatten, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Fixer les seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 1. CHARGEMENT
X = pd.read_csv('x_train_final.csv')
y = pd.read_csv('y_train_final.csv')
y_true = y['p0q0']

def preprocess_features(df_in, reference_df=None):
    df = df_in.copy()
    # Nettoyage Outliers
    for col in ['p0q3', 'p0q4']:
        median_val = df[col].median() if reference_df is None else reference_df[col].median()
        df.loc[df[col] < -160, col] = median_val
    
    # Date features
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['stop'] = df['arret']
    return df

X_proc = preprocess_features(X)

# 2. GRAPH FEATURES (Construction une seule fois)
print("Construction du graphe...")
G = nx.DiGraph()
temp_combined = X.copy()
temp_combined['delay_target'] = y_true.values
for _, group in temp_combined.groupby(['train', 'date']):
    group = group.sort_values('arret')
    gares = group['gare'].values
    delays = group['delay_target'].values
    for i in range(len(gares) - 1):
        e = (gares[i], gares[i+1])
        if G.has_edge(*e):
            G.edges[e]['delay'] += delays[i+1]
            G.edges[e]['count'] += 1
        else:
            G.add_edge(*e, delay=delays[i+1], count=1)

def get_flow(Graph, target, depth):
    if target not in Graph or not list(Graph.predecessors(target)): return 0
    # BFS inverse pour capter l'influence en amont
    edges = nx.bfs_edges(Graph, source=target, depth_limit=depth, reverse=True)
    data = np.array([[Graph.edges[e[::-1]]['delay'], Graph.edges[e[::-1]]['count']] for e in edges])
    return (data[:, 0] / data[:, 1].sum()).sum() if data.size > 0 else 0

# Pré-calcul des flows pour toutes les gares
unique_gares = X['gare'].unique()
flow_dict = {i: {g: get_flow(G, g, i) for g in unique_gares} for i in range(1, 9)}
degree_dict = dict(G.degree())

def apply_graph_features(df):
    df['degree'] = df['gare'].map(degree_dict).fillna(0)
    for i in range(1, 9):
        df[f'flowavg{i}'] = df['gare'].map(flow_dict[i]).fillna(0)
    return df

X_proc = apply_graph_features(X_proc)

# 3. PREPARATION DEEP LEARNING
num_cols = ['p2q0','p3q0','p4q0','p0q2','p0q3','p0q4','stop','day_of_week','month','degree'] + [f'flowavg{i}' for i in range(1, 9)]

train_idx, test_idx = train_test_split(X_proc.index, test_size=0.15, random_state=SEED)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_proc.loc[train_idx, num_cols])
X_num_test = scaler.transform(X_proc.loc[test_idx, num_cols])

le = LabelEncoder()
gare_train = le.fit_transform(X_proc.loc[train_idx, 'gare'])
# Mapping sécurisé pour les gares de test
gare_mapping = {label: i for i, label in enumerate(le.classes_)}
gare_test = X_proc.loc[test_idx, 'gare'].apply(lambda x: gare_mapping.get(x, 0)).values

# 4. ARCHITECTURE DU MODELE (Optimisée)
def build_optimized_model():
    # Entrées
    input_num = Input(shape=(len(num_cols),))
    input_gare = Input(shape=(1,))
    
    # Embedding
    emb = Embedding(input_dim=len(le.classes_) + 1, output_dim=128)(input_gare)
    emb = Flatten()(emb)
    
    # Fusion
    x = Concatenate()([input_num, emb])
    
    # Couches denses avec Swish (souvent meilleur que ReLU pour la régression)
    x = Dense(512, activation='swish')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    x = Dense(256, activation='swish')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.1)(x)
    
    x = Dense(128, activation='swish')(x)
    x = Dense(64, activation='swish')(x)
    
    output = Dense(1)(x)
    
    model = tf.keras.Model(inputs=[input_num, input_gare], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mae')
    return model

nn_model = build_optimized_model()

# Training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

print("Entraînement du modèle...")
nn_model.fit(
    [X_num_train, gare_train], y_true.iloc[train_idx],
    validation_split=0.15,
    epochs=200,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

# 5. PREDICTION FINALE
print("Traitement du fichier test final...")
X_test_final = pd.read_csv('x_test_final.csv')
X_test_proc = preprocess_features(X_test_final, reference_df=X)
X_test_proc = apply_graph_features(X_test_proc)

X_test_final_num = scaler.transform(X_test_proc[num_cols])
X_test_final_gare = X_test_proc['gare'].apply(lambda x: gare_mapping.get(x, 0)).values

y_final_preds = nn_model.predict([X_test_final_num, X_test_final_gare])

# Sauvegarde
submission = pd.DataFrame({'p0q0': y_final_preds.flatten()}, index=X_test_final.index)
submission.to_csv('submissionV6.csv', index=True)
print("Fichier submissionV6 généré.")

Construction du graphe...
Entraînement du modèle...
Epoch 1/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.7298 - val_loss: 0.7030 - learning_rate: 0.0010
Epoch 2/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 0.6916 - val_loss: 0.6885 - learning_rate: 0.0010
Epoch 3/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.6799 - val_loss: 0.6750 - learning_rate: 0.0010
Epoch 4/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6728 - val_loss: 0.6712 - learning_rate: 0.0010
Epoch 5/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6678 - val_loss: 0.6657 - learning_rate: 0.0010
Epoch 6/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6628 - val_loss: 0.6626 - learning_rate: 0.0010
Epoch 7/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 0.6595 - val_loss: 0.6605 - learning_rate: 0.0010
Epoch 8/200
3767/3767 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6564 - val_loss: 0.6593 - learning_rate: 0.0010
Epoch 9/200
3767/3767 ━━━━━━━━━━━━━━